# Comparer tous les backends OCR sur les memes tickets (Kaggle T4)

Lance `scripts/evaluate_all_backends.py`. Paddle, PP-OCRv4, le VLM hybride, l'OCR-VLM maison,
Groq et Moondream rendent chacun un Ticket canonique, et tous sont notes avec les memes
metriques (dont read_acc) sur les memes 18 photos.

Un backend dont la dependance, le checkpoint ou la cle manque est saute : il ne fait pas
tomber la comparaison.

A attacher en Input : le bundle de code, les tickets francais et leurs labels, le checkpoint
hybride, ceux de l'OCR-VLM maison, et si on veut la colonne Moondream, ses poids `.mf`.

GPU T4, internet active. La cle `GROQ_API_KEY` va dans les Secrets, pas dans le notebook.

In [ ]:
# Les backends a evaluer. Seuls ceux listes ici sont installes ET lances, donc autant
# garder la liste courte.
#
# Un seul backend prend quelques minutes, la liste complete plutot une heure. Et un backend
# seul evite le conflit de dependances entre paddleocr et moondream.
BACKENDS = ["hybrid"]
# BACKENDS = ["paddle", "ppocrv4", "hybrid", "groq", "moondream", "ocrvlm"]

print("BACKENDS =", BACKENDS)

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# On materialise le bundle de code
import glob, os, shutil, zipfile
from pathlib import Path
WORK = Path("/kaggle/working/repo")
def materialize():
    zips = glob.glob("/kaggle/input/**/receipt_vlm_colab_bundle.zip", recursive=True)
    if zips:
        WORK.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(WORK)
        return
    hits = glob.glob("/kaggle/input/**/vlm_training/scripts/evaluate_all_backends.py", recursive=True)
    if hits:
        dest = WORK / "dev_ocr"
        if not dest.exists():
            shutil.copytree(Path(hits[0]).resolve().parents[2], dest)
        return
    raise FileNotFoundError("code bundle not found in /kaggle/input -- Add Input the rebuilt bundle")
if not list(WORK.glob("**/vlm_training/scripts/evaluate_all_backends.py")):
    materialize()
hits = glob.glob(str(WORK / "**/vlm_training/scripts/evaluate_all_backends.py"), recursive=True)
assert hits, "evaluate_all_backends.py not found -- rebuild the bundle (scripts/zip_selfcontained_colab.py) and re-upload"
TRAIN_PKG = Path(hits[0]).resolve().parents[1]
DEV_OCR = TRAIN_PKG.parent
os.chdir(TRAIN_PKG)
print("Train package:", TRAIN_PKG)

In [ ]:
# On n'installe que ce dont les backends selectionnes ont besoin.
#
# Tout installer d'un coup ne marche pas : paddleocr et moondream tirent chacun leurs
# versions de huggingface_hub et de tokenizers, et elles s'excluent. D'ou le conditionnel.
import subprocess, sys

def pip(*a):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])

need = set(BACKENDS)

# Toujours : les deps d'entrainement, avec un trio de versions qui tiennent ensemble.
pip("-r", "requirements-training.txt",
    "transformers>=4.44,<5", "tokenizers>=0.22,<=0.23", "huggingface_hub>=0.30,<1.0")

if need & {"paddle", "ppocrv4"}:
    pip("paddleocr", "paddlepaddle")      # bloque huggingface_hub en version basse
if "groq" in need:
    pip("groq")
if "moondream" in need:
    try:
        pip("moondream==0.0.6")           # redescend tokenizers en 0.20.3
    except Exception as _e:
        print("moondream install skipped:", _e)

pip("-e", str(DEV_OCR))      # receipt_ocr : les backends et le parser
pip("-e", str(TRAIN_PKG))    # receipt_vlm : les deux modeles

if "hybrid" in need and (need & {"paddle", "ppocrv4", "moondream"}):
    print("WARNING: 'hybrid' together with paddle/ppocrv4/moondream re-introduces the")
    print("         transformers/huggingface_hub/tokenizers conflict that made it crash.")
    print("         Run BACKENDS = ['hybrid'] alone to get its numbers.")

import transformers, huggingface_hub, tokenizers
print("transformers", transformers.__version__,
      "| huggingface_hub", huggingface_hub.__version__,
      "| tokenizers", tokenizers.__version__)
print("Install OK for:", sorted(need))


In [ ]:
# Les datasets attaches, et la cle Groq
import glob, os
def _find(pattern):
    hits = glob.glob(pattern, recursive=True)
    return sorted(hits)[-1] if hits else None
def _find_dir_with(child):
    for d in glob.glob("/kaggle/input/**/" + child, recursive=True):
        if os.path.isdir(d):
            return os.path.dirname(d)
    return None
# Le jeu de test francais : le dossier qui contient real_labels/
FR_BASE = _find_dir_with("real_labels")
FR_IMAGES = os.path.join(FR_BASE, "images_tickets_caisse") if FR_BASE else None
FR_LABELS = os.path.join(FR_BASE, "real_labels") if FR_BASE else None
HYBRID_CKPT = _find("/kaggle/input/**/receipt_vlm_500m_merged.pt")
OCRVLM_CKPT = _find("/kaggle/input/**/ocr_vlm_epoch*.pt")
OCRVLM_TOK  = _find("/kaggle/input/**/tokenizer.json")
MOON_DIR    = _find_dir_with("*.mf") or None   # optionnel : None si les poids ne sont pas la
# La cle Groq vient des Secrets Kaggle, jamais du notebook
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")
    print("Groq key: loaded")
except Exception as e:
    print("Groq key: not set (", e, ") -> groq column will skip")
print("FR_IMAGES  ->", FR_IMAGES)
print("FR_LABELS  ->", FR_LABELS)
print("HYBRID     ->", HYBRID_CKPT)
print("OCRVLM     ->", OCRVLM_CKPT)
print("OCRVLM_TOK ->", OCRVLM_TOK)
print("MOONDREAM  ->", MOON_DIR)
assert FR_IMAGES and FR_LABELS, "attach the receipt-vlm-french-eval dataset (images_tickets_caisse + real_labels)"

if "hybrid" in BACKENDS:
    assert HYBRID_CKPT, "BACKENDS has 'hybrid' -> attach receipt-vlm-hybrid-ckpt (receipt_vlm_500m_merged.pt)"


In [ ]:
# Chaque backend tourne dans son propre sous-processus : sinon la memoire ne se libere
# pas entre deux, et le troisieme tombe.
import subprocess, sys, os
os.environ["RECEIPT_OCR_MAX_IMAGE_SIDE"] = "1024"   # sinon Paddle explose la memoire sur une photo de telephone
os.environ["OMP_NUM_THREADS"] = "2"
assert "BACKENDS" in dir(), "run the config cell first"
common = ["--french-images", FR_IMAGES, "--french-labels", FR_LABELS]
if HYBRID_CKPT: common += ["--hybrid-checkpoint", HYBRID_CKPT]
if OCRVLM_CKPT: common += ["--ocrvlm-checkpoint", OCRVLM_CKPT]
if OCRVLM_TOK:  common += ["--ocrvlm-tokenizer", OCRVLM_TOK]
if MOON_DIR:    common += ["--moondream-weights", MOON_DIR]
PARTIALS = []
for b in BACKENDS:
    print()
    print("################  " + b + "  ################", flush=True)
    out = "/kaggle/working/cmp_" + b + ".json"
    r = subprocess.run([sys.executable, "-u", "scripts/evaluate_all_backends.py",
                        "--backends", b, *common, "--output", out])
    if r.returncode == 0 and os.path.exists(out):
        PARTIALS.append(out)
    else:
        print("   [" + b + "] subprocess failed (rc " + str(r.returncode) + ") -> skipped", flush=True)
print()
print("collected:", PARTIALS)

In [ ]:
# On fusionne les resultats en une seule table
import json
ROWS = [("read_acc","Read acc (1-CER)"), ("valid","Valid (non-empty)"), ("product_recall","Product recall"),
        ("field_f1","Field F1"), ("anls","ANLS"), ("price_mae","Price MAE"), ("date_accuracy","Date exact match")]
results = []
for f in PARTIALS:
    for name, m in json.load(open(f)).get("results", {}).items():
        results.append((name, m))
names = [n for n, _ in results]
widths = [18] + [max(12, len(n)) for n in names]
print("  ".join(h.ljust(w) for h, w in zip(["Metric", *names], widths)))
print("-" * (sum(widths) + 2 * len(widths)))
for key, label in ROWS:
    cells = [label.ljust(widths[0])]
    for (_, m), w in zip(results, widths[1:]):
        cells.append(("%.3f" % m[key]).ljust(w))
    print("  ".join(cells))
print("n".ljust(widths[0]) + "  " + "  ".join(str(int(m["n_samples"])).ljust(w) for (_, m), w in zip(results, widths[1:])))